In [1]:
# ============================================================
# CELL 1 - Installations and Imports
# Run this cell first every time you open this notebook
# It installs all libraries and imports everything needed
# ============================================================

# Install all required libraries
# The ! means "run this as a terminal command"
# quiet flag (-q) reduces installation output noise
!pip install openai tavily-python biopython python-docx -q

# ── Imports ──────────────────────────────────────────────────

# OpenAI - to talk to GPT
from openai import OpenAI

# Tavily - for web search
from tavily import TavilyClient

# Biopython - for PubMed search
from Bio import Entrez

# python-docx - for creating Word documents
from docx import Document
from docx.shared import Inches, Pt

# Built in Python libraries - no installation needed
import json        # for handling tool call arguments
import getpass     # for safely entering API keys
import os          # for file and folder operations
from datetime import datetime  # for adding dates to filenames

print("✅ All libraries installed and imported successfully!")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ All libraries installed and imported successfully!


In [2]:
# ============================================================
# CELL 2 - API Keys
# Run this cell once at the start of every session
# Your keys are never saved in the file - only in memory
# ============================================================

# OpenAI API key - for GPT
openai_key = getpass.getpass("Paste your OpenAI API key: ")

# Tavily API key - for web search
tavily_key = getpass.getpass("Paste your Tavily API key: ")

# PubMed just needs your email - no key required
pubmed_email = "bansalneha2511@gmail.com"

# ── Create clients ───────────────────────────────────────────

# OpenAI client - connection to GPT
client = OpenAI(api_key=openai_key)

# Tavily client - connection to web search
tavily_client = TavilyClient(api_key=tavily_key)

# PubMed - just set the email
Entrez.email = pubmed_email

print("✅ All API keys loaded and clients ready!")

✅ All API keys loaded and clients ready!


In [3]:
# ============================================================
# CELL 3 - RWE Research Assistant System Prompt
# This defines how GPT behaves throughout every conversation
# It is specifically designed for Real World Evidence research
# ============================================================

SYSTEM_PROMPT = """
You are an expert Real World Evidence (RWE) Research Assistant 
specializing in healthcare and pharmaceutical research.

YOUR ROLE:
You help researchers find, summarize, and analyze published 
evidence from real world databases and scientific literature.

RESEARCH RULES - always follow these:
1. Always use the PubMed search tool for any medical, clinical,
   or scientific questions. Never answer from memory alone.
   
2. Always use web search for current news, general information,
   or anything not related to medical research.

3. Never make up citations, paper titles, authors or statistics.
   Only report what you actually find through your tools.
   
4. Always clearly state if findings across papers are 
   consistent or conflicting.

5. Always explain medical terms in simple language after 
   using them so non-experts can understand.

OUTPUT RULES - always structure your response like this:
1. Summary table of papers found
   (Title, Authors, Journal, Year, Key Finding)
   
2. Overall summary of findings in simple plain English
   (2-3 paragraphs maximum)
   
3. Consistency analysis
   (Are findings consistent or conflicting across papers?)
   
4. Limitations
   (What are the gaps in the evidence found?)

TONE:
- Clear and simple - avoid unnecessary jargon
- Thorough but concise - do not repeat yourself
- Professional - suitable for pharmaceutical research teams
- Honest - always flag uncertainty or missing information
"""

print("✅ RWE System Prompt ready!")

✅ RWE System Prompt ready!


In [4]:
# ============================================================
# CELL 4 - Conversation History
# Stores every message in the conversation
# Reset this whenever you start a new research question
# ============================================================

conversation_history = []

print("✅ Conversation history initialized!")
print(f"Messages in history: {len(conversation_history)}")

✅ Conversation history initialized!
Messages in history: 0


In [5]:
# ============================================================
# CELL 5 - Calculator Tool
# Performs accurate math calculations
# Prevents GPT from guessing and getting math wrong
# ============================================================

def calculator(expression):
    """
    Calculates a math expression accurately using Python.
    
    Args:
        expression → math problem as text e.g. "2348 * 4721"
    
    Returns:
        the correct answer as text
    """
    
    # eval() runs the math expression as real Python code
    # This is always accurate - no hallucination possible
    result = eval(expression)
    
    return str(result)

# Quick test to make sure it works
print("✅ Calculator tool ready!")
print(f"Quick test - 2348 * 4721 = {calculator('2348 * 4721')}")

✅ Calculator tool ready!
Quick test - 2348 * 4721 = 11084908


In [6]:
# ============================================================
# CELL 6 - PubMed Search Tool
# Searches 35 million+ real verified published papers
# Returns actual titles, authors, journals and abstracts
# Zero hallucination - everything comes from real papers
# ============================================================

def search_pubmed(query):
    """
    Searches PubMed for real published research papers.
    
    Args:
        query → research search term 
                e.g. "ESR1 mutation metastatic breast cancer"
    
    Returns:
        real paper details including titles, authors, 
        journals, years and abstracts
    """

    # STEP 1 - Search PubMed for paper IDs matching our query
    # retmax=5 limits to 5 papers to keep token usage low
    search_handle = Entrez.esearch(
        db="pubmed",    # search the PubMed database
        term=query,     # our research query
        retmax=5        # maximum 5 papers
    )

    # Read results and close connection
    search_results = Entrez.read(search_handle)
    search_handle.close()

    # Get the list of paper IDs found
    paper_ids = search_results["IdList"]

    # If nothing found return a clear message
    if not paper_ids:
        return f"No papers found on PubMed for: {query}"

    # STEP 2 - Fetch full details for each paper ID
    fetch_handle = Entrez.efetch(
        db="pubmed",
        id=paper_ids,        # IDs from step 1
        rettype="abstract",  # get abstract level detail
        retmode="text"       # return as plain text
    )

    # Read the full paper details
    papers_text = fetch_handle.read()
    fetch_handle.close()

    return papers_text

print("✅ PubMed search tool ready!")

✅ PubMed search tool ready!


In [7]:
# ============================================================
# CELL 7 - Web Search Tool
# Searches the web for current general information
# Use this for non-medical queries only
# For medical research always use PubMed instead
# ============================================================

def search_web(query):
    """
    Searches the web for current general information.
    
    Args:
        query → search term e.g. "latest news in pharma India"
    
    Returns:
        search results as text
    """

    # Send query to Tavily and get results back
    results = tavily_client.search(query)

    # Combine all results into one clean text
    # Each result has a title and content
    search_results = ""
    for result in results["results"]:
        search_results += f"Title: {result['title']}\n"
        search_results += f"Content: {result['content']}\n\n"

    return search_results

print("✅ Web search tool ready!")

✅ Web search tool ready!


In [8]:
# ============================================================
# CELL 8 - Tools List
# Describes all available tools to GPT
# GPT reads these descriptions and decides which tool to use
# The descriptions are critical - they guide GPT's decisions
# ============================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": """Use this tool for any math calculation. 
            Always use this instead of calculating yourself. 
            Use Python operators: + for addition, - for subtraction, 
            * for multiplication, / for division, 
            ** for power/raised to (NOT ^), % for remainder""",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "the math expression to calculate using Python operators, for example 2348 * 4721 or 2 ** 10 for powers"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_pubmed",
            "description": """Use this tool for ANY medical, clinical, 
            scientific or research related queries. 
            This searches 35 million real verified published papers on PubMed. 
            Always use this instead of web search for: finding references, 
            citations, research findings, clinical studies, drug information, 
            mutation rates, disease statistics, or any healthcare and 
            pharma related questions. 
            Use web search only for general non-medical information.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "the medical or research search term, for example 'ESR1 mutation rates metastatic breast cancer'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": """Use this tool to search the web for current 
            general information, news, weather, or anything that requires 
            up to date information that is NOT medical or scientific research.
            For any medical, clinical or research queries always use 
            search_pubmed instead.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "the general search query, for example 'latest pharma news India' or 'current weather Delhi'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("✅ All 3 tools ready!")
print("Tools available: calculator, search_pubmed, search_web")

✅ All 3 tools ready!
Tools available: calculator, search_pubmed, search_web


In [9]:
# ============================================================
# CELL 9 - Agent Loop
# The heart of the agent
# Keeps running until GPT gives a final answer
# or until max steps is reached for safety
# ============================================================

def run_agent(user_message):
    """
    Runs the agent loop for a given research question.
    
    Args:
        user_message → your research question as text
    
    Returns:
        GPT's final answer after using all necessary tools
    """

    # Add user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Safety limit - prevents infinite loops
    max_steps = 10
    current_step = 0

    # ── THE AGENT LOOP ───────────────────────────────────────
    while current_step < max_steps:

        current_step += 1
        print(f"\n── Step {current_step} ──")

        # Ask GPT what to do next
        # Send full history + all tools every time
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                *conversation_history
            ],
            tools=tools
        )

        gpt_response = response.choices[0].message

        # ── DID GPT WANT TO USE A TOOL? ──────────────────────
        if gpt_response.tool_calls:

            # Add GPT's tool request to history first
            conversation_history.append({
                "role": "assistant",
                "content": str(gpt_response.content or ""),
                "tool_calls": gpt_response.tool_calls
            })

            # Loop through ALL tool calls GPT requested
            # GPT might request multiple tools at once
            for tool_call in gpt_response.tool_calls:

                tool_name = tool_call.function.name
                tool_input = json.loads(tool_call.function.arguments)

                print(f"GPT is using: {tool_name}")
                print(f"Query: {tool_input}")

                # Run the correct tool
                if tool_name == "calculator":
                    tool_result = calculator(tool_input["expression"])
                elif tool_name == "search_pubmed":
                    tool_result = search_pubmed(tool_input["query"])
                elif tool_name == "search_web":
                    tool_result = search_web(tool_input["query"])

                print(f"✅ Tool result received")

                # Add tool result to history
                # Linked to its specific tool call ID
                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                })

        else:
            # ── GPT GAVE A FINAL ANSWER ──────────────────────
            final_reply = gpt_response.content

            # Add final reply to history
            conversation_history.append({
                "role": "assistant",
                "content": final_reply
            })

            print(f"\n✅ Agent completed in {current_step} steps!")
            return final_reply

    # If max steps reached
    return "Max steps reached without a final answer."

print("✅ Agent loop ready!")

✅ Agent loop ready!


In [10]:
# ============================================================
# CELL 10 - Word Document Export Function
# Automatically saves research findings as a Word document
# Creates a professionally formatted .docx file
# ============================================================

def export_to_word(research_question, findings):
    """
    Exports research findings to a Word document.
    
    Args:
        research_question → the question you asked the agent
        findings          → GPT's full research answer
    
    Returns:
        the file path where the document was saved
    """

    # Create a new Word document
    doc = Document()

    # ── TITLE ────────────────────────────────────────────────
    # Add the main title at the top
    title = doc.add_heading("RWE Research Findings", level=1)

    # ── DATE AND TIME ─────────────────────────────────────────
    # Add the date and time the report was generated
    # datetime.now() gets the current date and time
    # strftime formats it nicely e.g. "25 May 2026 14:30"
    date_str = datetime.now().strftime("%d %B %Y %H:%M")
    doc.add_paragraph(f"Generated: {date_str}")

    # Add a horizontal line for visual separation
    doc.add_paragraph("─" * 60)

    # ── RESEARCH QUESTION ─────────────────────────────────────
    # Add the research question as a heading
    doc.add_heading("Research Question", level=2)
    doc.add_paragraph(research_question)

    # ── FINDINGS ──────────────────────────────────────────────
    # Add the full research findings
    doc.add_heading("Research Findings", level=2)
    doc.add_paragraph(findings)

    # ── FOOTER ────────────────────────────────────────────────
    # Add a disclaimer at the bottom
    doc.add_paragraph("─" * 60)
    doc.add_paragraph(
        "Generated by RWE Research Agent | "
        "Always verify findings independently "
        "before use in publications or protocols."
    )

    # ── SAVE THE FILE ─────────────────────────────────────────
    # Create a filename using the date and time
    # so every report has a unique name
    # e.g. "RWE_Research_20260525_1430.docx"
    filename = f"RWE_Research_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"

    # Save in the current folder
    doc.save(filename)

    return filename

print("✅ Word export function ready!")

✅ Word export function ready!


In [13]:
# ============================================================
# CELL 10 - Word Document Export Function (Updated)
# Now properly formats markdown tables as real Word tables
# Creates a professionally formatted .docx file
# ============================================================

from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

def is_table_row(line):
    """
    Checks if a line of text is part of a markdown table.
    A table row starts and ends with | character.
    
    Args:
        line → one line of text
    
    Returns:
        True if it is a table row, False if not
    """
    # strip() removes spaces from start and end
    # then we check if line starts and ends with |
    stripped = line.strip()
    return stripped.startswith("|") and stripped.endswith("|")

def is_separator_row(line):
    """
    Checks if a line is the separator row in a markdown table.
    Separator rows look like: |---|---|---|
    
    Args:
        line → one line of text
    
    Returns:
        True if it is a separator row, False if not
    """
    stripped = line.strip()
    # replace removes all - and | and spaces
    # if nothing is left it was a separator row
    return is_table_row(line) and stripped.replace("-", "").replace("|", "").replace(" ", "") == ""

def parse_table_row(line):
    """
    Converts a markdown table row into a list of cell values.
    
    Example:
        "| Title | Authors | Year |"
        becomes → ["Title", "Authors", "Year"]
    
    Args:
        line → one markdown table row as text
    
    Returns:
        list of cell values with spaces stripped
    """
    # split by | to get each cell
    # [1:-1] removes the empty first and last items
    # strip() removes extra spaces from each cell
    cells = line.strip().split("|")
    return [cell.strip() for cell in cells[1:-1]]

def add_table_to_doc(doc, table_lines):
    """
    Creates a real Word table from markdown table lines.
    
    Args:
        doc         → the Word document object
        table_lines → list of markdown table row strings
    """

    # Filter out separator rows - we don't need them in Word
    # They are just visual separators in markdown
    data_rows = [line for line in table_lines if not is_separator_row(line)]

    # If no data found return without doing anything
    if not data_rows:
        return

    # Parse all rows into lists of cell values
    parsed_rows = [parse_table_row(row) for row in data_rows]

    # Get number of columns from first row (header row)
    num_cols = len(parsed_rows[0])
    num_rows = len(parsed_rows)

    # Create the Word table with correct dimensions
    table = doc.add_table(rows=num_rows, cols=num_cols)

    # Style the table - this gives it clean borders
    table.style = "Table Grid"

    # Fill in the table cells row by row
    for row_idx, row_data in enumerate(parsed_rows):
        for col_idx, cell_value in enumerate(row_data):

            # Get the cell at this position
            cell = table.cell(row_idx, col_idx)

            # Set the cell text
            cell.text = cell_value

            # Make the header row bold
            # Header is always the first row (row_idx == 0)
            if row_idx == 0:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.bold = True

def export_to_word(research_question, findings):
    """
    Exports research findings to a properly formatted Word document.
    Converts markdown tables into real Word tables automatically.
    
    Args:
        research_question → the question you asked the agent
        findings          → GPT's full research answer
    
    Returns:
        the filename where the document was saved
    """

    # Create a new Word document
    doc = Document()

    # ── TITLE ─────────────────────────────────────────────────
    doc.add_heading("RWE Research Findings", level=1)

    # ── DATE AND TIME ──────────────────────────────────────────
    date_str = datetime.now().strftime("%d %B %Y %H:%M")
    doc.add_paragraph(f"Generated: {date_str}")
    doc.add_paragraph("─" * 60)

    # ── RESEARCH QUESTION ──────────────────────────────────────
    doc.add_heading("Research Question", level=2)
    doc.add_paragraph(research_question)

    # ── FINDINGS ───────────────────────────────────────────────
    doc.add_heading("Research Findings", level=2)

    # Split findings into individual lines for processing
    lines = findings.split("\n")

    # We process line by line
    # When we detect a table we collect all its rows
    # then convert them to a real Word table
    table_lines = []      # stores current table rows
    in_table = False      # tracks if we are inside a table

    for line in lines:

        if is_table_row(line):
            # This line is part of a table
            # Add it to our collection
            in_table = True
            table_lines.append(line)

        else:
            # This line is NOT a table row
            # If we were collecting a table, convert it now
            if in_table and table_lines:
                add_table_to_doc(doc, table_lines)
                table_lines = []    # reset for next table
                in_table = False
                doc.add_paragraph("")  # add space after table

            # Skip empty lines and markdown heading markers
            # but add them as spacing
            if line.strip() == "":
                doc.add_paragraph("")

            elif line.strip().startswith("###"):
                # ### means heading level 3 in markdown
                # remove the ### and add as Word heading
                heading_text = line.strip().replace("###", "").strip()
                doc.add_heading(heading_text, level=3)

            elif line.strip().startswith("##"):
                # ## means heading level 2 in markdown
                heading_text = line.strip().replace("##", "").strip()
                doc.add_heading(heading_text, level=2)

            elif line.strip().startswith("#"):
                # # means heading level 1 in markdown
                heading_text = line.strip().replace("#", "").strip()
                doc.add_heading(heading_text, level=1)

            else:
                # Regular paragraph text
                # Remove markdown bold markers (**)
                clean_line = line.replace("**", "")
                if clean_line.strip():
                    doc.add_paragraph(clean_line)

    # If document ends with a table make sure we add it
    if in_table and table_lines:
        add_table_to_doc(doc, table_lines)

    # ── FOOTER ────────────────────────────────────────────────
    doc.add_paragraph("─" * 60)
    doc.add_paragraph(
        "Generated by RWE Research Agent | "
        "Always verify findings independently "
        "before use in publications or protocols."
    )

    # ── SAVE FILE ─────────────────────────────────────────────
    filename = f"RWE_Research_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"
    doc.save(filename)

    return filename

print("✅ Word export function updated with proper table formatting!")

✅ Word export function updated with proper table formatting!


In [14]:
# ============================================================
# CELL 11 - Full RWE Research Agent
# This is the main function that ties everything together
# One function call does everything:
# 1. Runs the research
# 2. Exports to Word document
# 3. Tells you where the file was saved
# ============================================================

def rwe_agent(research_question):
    """
    The main RWE Research Agent function.
    Call this with any research question and it will:
    1. Search PubMed for real papers
    2. Summarize findings in a structured format
    3. Export everything to a Word document
    
    Args:
        research_question → your research question as text
    
    Returns:
        prints the findings and saves a Word document
    """

    print("=" * 60)
    print("RWE RESEARCH AGENT")
    print("=" * 60)
    print(f"Research Question: {research_question}")
    print("=" * 60)

    # STEP 1 - Reset conversation history
    # Fresh start for every new research question
    global conversation_history
    conversation_history = []

    # STEP 2 - Run the agent loop
    # This searches PubMed and generates findings
    print("\nSearching and analyzing evidence...")
    findings = run_agent(research_question)

    # STEP 3 - Print the findings
    print("\n" + "=" * 60)
    print("RESEARCH FINDINGS")
    print("=" * 60)
    print(findings)

    # STEP 4 - Export to Word document
    print("\n" + "=" * 60)
    print("Exporting to Word document...")
    filename = export_to_word(research_question, findings)

    print(f"✅ Report saved as: {filename}")
    print("=" * 60)

print("✅ RWE Research Agent ready!")
print("\nUsage:")
print('rwe_agent("your research question here")')

✅ RWE Research Agent ready!

Usage:
rwe_agent("your research question here")


In [15]:
rwe_agent("What are the real world treatment patterns and outcomes in ESR1 mutated metastatic breast cancer?")

RWE RESEARCH AGENT
Research Question: What are the real world treatment patterns and outcomes in ESR1 mutated metastatic breast cancer?

Searching and analyzing evidence...

── Step 1 ──
GPT is using: search_pubmed
Query: {'query': 'ESR1 mutated metastatic breast cancer treatment patterns outcomes'}
✅ Tool result received

── Step 2 ──

✅ Agent completed in 2 steps!

RESEARCH FINDINGS
### Summary Table of Papers Found

| Title                                                                                                        | Authors                                                                                                             | Journal                           | Year | Key Finding                                                                                                                                                                                                                                                                                        |
|---------

In [21]:
rwe_agent("can you look at the kundali as well? if yes, kindly check the horsoscope of Vaibhav Bansal botn born on 26Feb 1996 at 1:40 pm new delhi and see when will he get married?")

RWE RESEARCH AGENT
Research Question: can you look at the kundali as well? if yes, kindly check the horsoscope of Vaibhav Bansal botn born on 26Feb 1996 at 1:40 pm new delhi and see when will he get married?

Searching and analyzing evidence...

── Step 1 ──
GPT is using: search_web
Query: {'query': 'Vaibhav Bansal horoscope marriage prediction February 26 1996'}
✅ Tool result received

── Step 2 ──

✅ Agent completed in 2 steps!

RESEARCH FINDINGS
I'm unable to provide horoscope readings or predictions regarding specific individual events such as marriage. You might want to consult a professional astrologer for personalized insights based on the individual’s astrological chart.

Exporting to Word document...
✅ Report saved as: RWE_Research_20260528_1949.docx
